In [1]:
from pathlib import Path
import duckdb
from sklearn.model_selection import StratifiedKFold
import pandas as pd
from itertools import product
import pandera.pandas as pa
import pyarrow
import os
import sys
sys.path.append(str(Path(os.getcwd()).parent.absolute()))

import colorlog
logger = colorlog.getLogger()

In [2]:
BASE_QEUERY = """
SELECT 
    id,
    target,
    el_lhmedium,
    el_lhvloose,
    IF({et_col} < 0, 0, {et_col}) AS {et_col}_normalized,
    ABS({eta_col}) AS {eta_col}_normalized,
    CASE WHEN el_lhmedium AND target=1 THEN TRUE
         WHEN NOT el_lhvloose AND target=0 THEN FALSE
         ELSE NULL END AS standard_binning_label,
FROM read_parquet('{table_data_path}')
WHERE {et_col}_normalized >= {et_bin_left} AND
      {et_col}_normalized < {et_bin_right} AND
      {eta_col}_normalized >= {eta_bin_left} AND
      {eta_col}_normalized < {eta_bin_right} AND
      standard_binning_label IS NOT NULL;
"""

RESULT_QUERY = """
SELECT
    id,
    CASE WHEN el_lhmedium AND target=1 THEN TRUE
         WHEN NOT el_lhvloose AND target=0 THEN FALSE
         ELSE NULL END AS standard_binning_label,
    NULL AS standard_binning_kfold
FROM read_parquet('{table_data_path}');
"""

kfold_dataframe_schema = pa.DataFrameSchema({
    'id': pa.Column(
        pd.ArrowDtype(pyarrow.uint64()),
        unique=True,
        checks=[
            pa.Check.ge(0),
        ]),
    'standard_binning_label': pa.Column(
        pd.ArrowDtype(pyarrow.bool_()),
        nullable=True),
    'standard_binning_kfold': pa.Column(
        pd.ArrowDtype(pyarrow.uint8()),
        nullable=True,
        checks=[
            pa.Check.ge(0),
            pa.Check.le(4),
        ])
})

In [3]:
dataset_dir = Path('/home/lucasbanunes/workspaces/ringer-zero/tests/data/test_dataset')
et_bins = [15, 20, 30, 40, 50, int(1e12)]
et_col = 'trig_L2_calo_et'
eta_bins = [0, 0.8, 1.37, 1.52, 2.500001]
eta_col = 'trig_L2_calo_eta'
splitter = StratifiedKFold(n_splits=5, shuffle=True)

In [4]:
result_query = RESULT_QUERY.format(
    table_data_path=str(dataset_dir / 'data.parquet' / 'data_*.parquet')
)
results_df = duckdb.query(result_query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
results_df

,id,standard_binning_label,standard_binning_kfold
0,0,False,<NA>
1,1,False,<NA>
2,2,<NA>,<NA>
3,3,<NA>,<NA>
4,4,<NA>,<NA>
...,...,...,...
199995,199995,<NA>,<NA>
199996,199996,True,<NA>
199997,199997,<NA>,<NA>
199998,199998,False,<NA>


In [5]:
results_df.dtypes

id                        uint64[pyarrow]
standard_binning_label      bool[pyarrow]
standard_binning_kfold     int32[pyarrow]
dtype: object

In [6]:
bins_iterator = product(zip(et_bins[:-1], et_bins[1:]), zip(eta_bins[:-1], eta_bins[1:]))
for (et_bin_left, et_bin_right), (eta_bin_left, eta_bin_right) in bins_iterator:
    print(f'Processing bin: ET in [{et_bin_left}, {et_bin_right}), ETA in [{eta_bin_left}, {eta_bin_right})')
    current_query = BASE_QEUERY.format(
        et_col=et_col,
        eta_col=eta_col,
        table_data_path=str(dataset_dir / 'data.parquet' / 'data_*.parquet'),
        et_bin_left=int(et_bin_left*1e3),
        et_bin_right=int(et_bin_right*1e3),
        eta_bin_left=eta_bin_left,
        eta_bin_right=eta_bin_right
    )
    logger.debug(f'Executing query:\n{current_query}')
    df = duckdb.query(current_query).fetch_arrow_table().to_pandas(types_mapper=pd.ArrowDtype)
    id_col = df['id'].astype(int)
    label_col = df['standard_binning_label'].astype(int)
    for fold, (_, val_idx) in enumerate(splitter.split(id_col.values, label_col.values)):
        print(f'Assigning fold {fold} to {len(val_idx)} samples')
        val_ids = df['id'].iloc[val_idx]
        results_df.loc[results_df['id'].isin(val_ids), 'standard_binning_kfold'] = fold

Processing bin: ET in [15, 20), ETA in [0, 0.8)
Assigning fold 0 to 373 samples
Assigning fold 1 to 373 samples
Assigning fold 2 to 373 samples
Assigning fold 3 to 373 samples
Assigning fold 4 to 373 samples
Processing bin: ET in [15, 20), ETA in [0.8, 1.37)
Assigning fold 0 to 266 samples
Assigning fold 1 to 265 samples
Assigning fold 2 to 265 samples
Assigning fold 3 to 265 samples
Assigning fold 4 to 265 samples
Processing bin: ET in [15, 20), ETA in [1.37, 1.52)
Assigning fold 0 to 75 samples
Assigning fold 1 to 75 samples
Assigning fold 2 to 75 samples
Assigning fold 3 to 74 samples
Assigning fold 4 to 74 samples
Processing bin: ET in [15, 20), ETA in [1.52, 2.500001)
Assigning fold 0 to 442 samples
Assigning fold 1 to 442 samples
Assigning fold 2 to 442 samples
Assigning fold 3 to 442 samples
Assigning fold 4 to 442 samples
Processing bin: ET in [20, 30), ETA in [0, 0.8)
Assigning fold 0 to 717 samples
Assigning fold 1 to 716 samples
Assigning fold 2 to 716 samples
Assigning fold

In [7]:
results_df['standard_binning_kfold'] = results_df['standard_binning_kfold'].astype('uint8[pyarrow]')
results_df

,id,standard_binning_label,standard_binning_kfold
0,0,False,2
1,1,False,<NA>
2,2,<NA>,<NA>
3,3,<NA>,<NA>
4,4,<NA>,<NA>
...,...,...,...
199995,199995,<NA>,<NA>
199996,199996,True,2
199997,199997,<NA>,<NA>
199998,199998,False,4


In [8]:
results_df.dtypes

id                        uint64[pyarrow]
standard_binning_label      bool[pyarrow]
standard_binning_kfold     uint8[pyarrow]
dtype: object

# Validate

In [9]:
results_df = kfold_dataframe_schema.validate(results_df)
results_df

,id,standard_binning_label,standard_binning_kfold
0,0,False,2
1,1,False,<NA>
2,2,<NA>,<NA>
3,3,<NA>,<NA>
4,4,<NA>,<NA>
...,...,...,...
199995,199995,<NA>,<NA>
199996,199996,True,2
199997,199997,<NA>,<NA>
199998,199998,False,4


# Saving data

In [10]:
sample_per_batch = 100_000
output_dir = dataset_dir / 'standard_binning_kfold.parquet'
output_dir.mkdir(exist_ok=True)
for file_count, start in enumerate(range(0, len(results_df), sample_per_batch)):
    end = start + sample_per_batch
    if end > len(results_df):
        end = len(results_df)
    print(f'Saving batch {file_count} with samples from index {start} to {end}')
    batch_df = results_df.iloc[start:end]
    batch_path = output_dir / f'standard_binning_kfold_{file_count:04d}.parquet'
    batch_df.to_parquet(batch_path, index=False)

Saving batch 0 with samples from index 0 to 100000
Saving batch 1 with samples from index 100000 to 200000
